## Import libraries

In [1]:
!pip install -U torch torchvision torchaudio
!pip install -U langchain langchain-community langchain-core langchain-huggingface
!pip install -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu astunparse huggingface_hub pypdf langchain-community
!pip install "ipywidgets>=7,<8"
!pip install "datasets<4.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5

In [2]:
import os
import torch
import ast
import random
import json
import gc
import re
from datetime import datetime
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import shutil
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import load_dataset

Checking that we are using the GPU

In [3]:
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("No GPU detected. Please change runtime settings.")

GPU Detected: Tesla T4
Memory: 15.83 GB


Login to HuggingFace to access models

In [4]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## Chose the Llama-3-8B-Instruct for Socratic reasoning

https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct but
using the https://huggingface.co/unsloth/llama-3-8b-Instruct-bnb-4bit because is optimized

In [5]:
model_id = "unsloth/llama-3-8b-instruct-bnb-4bit"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

## Create the pipeline

In [6]:
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.6,
    top_p=0.9,
    repetition_penalty=1.1
)

Device set to use cuda:0


## Method that handles loading PDFs, chunking text, and searching via FAISS (Facebook AI Similarity Search)

In [7]:
class KnowledgeBase:
    def __init__(self):
        self.vector_store = None
        # we use a small, fast embedding model for the curriculum
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        self.is_loaded = False

    # reads a PDF or Text file and builds the index
    def ingest_file(self, file_path):
        if file_path.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        else:
            loader = TextLoader(file_path)

        docs = loader.load()

        # split into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )
        splits = text_splitter.split_documents(docs)

        # build vector index (FAISS)
        self.vector_store = FAISS.from_documents(splits, self.embeddings)
        self.is_loaded = True

    # find the top k relevant chunks for a query
    def search(self, query, k=3):
        if not self.vector_store:
            return "No curriculum loaded."

        results = self.vector_store.similarity_search(query, k=k)
        # format the context string
        context = "\n\n".join([f"[Source: Page {doc.metadata.get('page', 0)}]: {doc.page_content}" for doc in results])
        return context

Initialize knowledge base

In [8]:
rag_engine = KnowledgeBase()

/tmp/ipython-input-589872701.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Method to get programming exercises from the CodeParrot db

https://huggingface.co/datasets/codeparrot/apps

In [9]:
class ExerciseDB:
    def __init__(self):
        # we allow shuffling with a buffer to ensure randomness
        self.dataset = load_dataset("codeparrot/apps", split="train", trust_remote_code=True)
        self.shuffled_stream = self.dataset.shuffle(seed=random.randint(0, 1000))

    # fetch a random problem matching the difficulty ("Any", "introductory", "interview", "competition")
    def get_random_problem(self, difficulty_level="Any"):
        attempts = 0
        # re-shuffle occasionally to prevent repeating patterns
        iterator = iter(self.shuffled_stream)

        while attempts < 100:
            try:
                problem = next(iterator)
                attempts += 1

                # check filter
                if difficulty_level != "Any" and problem['difficulty'] != difficulty_level:
                    continue

                # found a match
                question = problem['question']
                try:
                    solutions = json.loads(problem['solutions'])
                    solution = solutions[0] if solutions else ""
                except:
                    solution = problem['solutions']

                return {
                    "id": problem['problem_id'],
                    "question": question,
                    "starter_code": problem['starter_code'] if problem['starter_code'] else "",
                    "solution": solution,
                    "difficulty": problem['difficulty']
                }
            except StopIteration:
                # if stream ends, re-init
                self.shuffled_stream = self.dataset.shuffle(seed=random.randint(0, 1000), buffer_size=1000)
                iterator = iter(self.shuffled_stream)

        return None

Initialize method

In [10]:
exercise_db = ExerciseDB()

README.md: 0.00B [00:00, ?B/s]

apps.py: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/107M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.29G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

## Method that manages chat history to prevent out of memory errors

In [11]:
class TutorSession:
    def __init__(self, log_file="chat_history.json"):
        # store full conversation for research
        self.log_file = log_file
        self.history = []
        self.stm_window = []
        self.attempt_count = 0
        self.puzzle_triggered = False
        self.active_problem_text = ""
        self.active_problem_solution = ""
        self._load_state()

        # initialize empty JSON if not exists
        if not os.path.exists(log_file):
            with open(log_file, 'w') as f:
                json.dump([], f)

    # method that restores history and attempts from the JSON file
    def _load_state(self):
        if os.path.exists(self.log_file):
            try:
                with open(self.log_file, 'r') as f:
                    data = json.load(f)
                    if isinstance(data, list) and len(data) > 0:
                        self.history = data

                        # find the last "NEW CHAT" marker
                        last_reset_index = -1
                        for i, entry in enumerate(reversed(data)):
                            if entry.get("role") == "system" and "NEW CHAT" in entry.get("content", ""):
                                last_reset_index = len(data) - 1 - i
                                break

                        # only load context AFTER the last reset
                        if last_reset_index != -1:
                            relevant_history = data[last_reset_index+1:]
                        else:
                            relevant_history = data

                        # restore attempt count from the very last entry
                        last_entry = self.history[-1]
                        self.attempt_count = last_entry.get("attempt_count", 0)
                        self.puzzle_triggered = last_entry.get("puzzle_triggered", False)

                        # restore short term memory (last 4 turns)
                        recent = relevant_history[-4:]
                        self.stm_window = [{"role": t["role"], "content": t["content"]} for t in recent if t["role"] != "system"]
            except Exception as e:
                print(f"Error loading history: {e}")

    def start_new_chat(self):
        self.stm_window = []
        self.attempt_count = 0
        self.puzzle_triggered = False
        self.active_problem_text = ""
        self.active_problem_solution = ""

        # log the reset so we don't load old context next time
        marker = {
            "timestamp": datetime.now().isoformat(),
            "role": "system",
            "content": "NEW CHAT SESSION STARTED",
            "attempt_count": 0,
            "puzzle_triggered": False
        }
        self.history.append(marker)
        self._save_to_file()

    def set_active_problem(self, question, solution):
        self.active_problem_text = question
        self.active_problem_solution = solution

        # reset session first
        self.stm_window = []
        self.attempt_count = 0
        self.puzzle_triggered = False

        marker = {
            "timestamp": datetime.now().isoformat(),
            "role": "system",
            "content": f"STARTED PROBLEM: {question[:100]}...",
            "attempt_count": 0,
            "puzzle_triggered": False
        }
        self.history.append(marker)
        self._save_to_file()

    def add_turn(self, role, content):
        # update short term memory (keep only last 4 messages to save VRAM)
        if role != "system":
            self.stm_window.append({"role": role, "content": content})
            if len(self.stm_window) > 4:
                self.stm_window.pop(0)

        # update long term research log
        entry = {
            "timestamp": datetime.now().isoformat(),
            "role": role,
            "content": content,
            "attempt_count": self.attempt_count,
            "puzzle_triggered": self.puzzle_triggered
        }
        self.history.append(entry)
        self._save_to_file()

    def increment_attempts(self):
        self.attempt_count += 1

    def reset_attempts(self):
        self.attempt_count = 0

    def mark_puzzle_as_given(self):
        self.puzzle_triggered = True

    def _save_to_file(self):
        # append the session to the JSON file
        with open(self.log_file, 'w') as f:
            json.dump(self.history, f, indent=2)

## Method to detect language

In [12]:
def detect_language(code_snippet):
    if re.search(r'#include\s+<', code_snippet) or re.search(r'std::', code_snippet):
        return "cpp"
    if re.search(r'public\s+class', code_snippet) or re.search(r'System\.out\.println', code_snippet):
        return "java"
    if re.search(r'def\s+\w+\(', code_snippet) or re.search(r'print\(', code_snippet):
        return "python"
    return "unknown"

## Static analysis (feedback ladder Level 1)

Adapts analysis based on language:
* Python: uses strict 'ast' parsing
* Others: returns 'Unknown' (the LLM will have to catch errors)

In [13]:
def analyze_code_structure(code_snippet):
    lang = detect_language(code_snippet)

    if lang == "python":
        try:
            ast.parse(code_snippet)
            return {"is_valid": True, "feedback": "Syntax is valid Python."}
        except SyntaxError as e:
            return {"is_valid": False, "feedback": f"Python Syntax Error on line {e.lineno}: {e.msg}"}

    return {"is_valid": True, "feedback": f"Static Analysis not available for {lang}. Please rely on your own knowledge"}

## Background solver agent method that fixes the student's code to create the puzzle solution

In [14]:
def generate_golden_solution(broken_code):
    torch.cuda.empty_cache()

    solver_prompt = f"""
    You are an expert programmer.
    TASK: Identify the programming language and fix the bugs in the following code.
    OUTPUT: Provide ONLY the raw corrected code. No markdown, no explanations, no comments.

    BROKEN CODE:
    {broken_code}
    """

    messages = [{"role": "user", "content": solver_prompt}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = text_generator(prompt, max_new_tokens=256, do_sample=False)

    # extract code
    raw_output = outputs[0]["generated_text"][len(prompt):].strip()

    # removing markdown blocks if the model adds them
    clean_code = re.sub(r'```[a-zA-Z]*|```', '', raw_output).strip()
    return clean_code

## Parsons puzzle generator (over-reliance guardrail)

In [15]:
def generate_parsons_puzzle(correct_code):
    lines = [line.strip() for line in correct_code.split('\n') if line.strip()]
    random.shuffle(lines)
    puzzle_text = "You seem stuck, so let's try a different approach. Instead of writing code, can you arrange these scrambled lines in the correct order?\n\n"
    for i, line in enumerate(lines):
        puzzle_text += f"{i+1}. <--> {line}\n"
    return puzzle_text

## Method that checks if the text contains markdown code blocks

In [16]:
def has_code_block(text):
    return "```" in text or "print(" in text or "def " in text

## Socratic pipeline with memory management - to enforce the "direct answer" trap

In [ ]:
def generate_tutor_response(student_code, user_query, session):
    # clear GPU Memory
    torch.cuda.empty_cache()
    gc.collect()

    has_code = len(student_code.strip()) > 10

    if has_code and rag_engine.is_loaded:
        context = rag_engine.search(user_query)
        system_instruction = f"""
        You are a Helpful Professor.
        Answer the student's question using ONLY the Curriculum Context below.
        If the answer is not in the context, say "I don't see that in the curriculum."

        CURRICULUM CONTEXT:
        {context}
        """
        is_theory = True

    else:
        is_theory = False
        # check for over-reliance (Parsons Trigger)
        session.increment_attempts()
        if session.attempt_count >= 5 and not session.puzzle_triggered:
            session.reset_attempts()
            session.mark_puzzle_as_given()
            if session.active_problem_solution:
                # using solution from dataset
                solution_code = session.active_problem_solution
            else:
                solution_code = generate_golden_solution(student_code)
            return generate_parsons_puzzle(solution_code)

        # static analysis of the code
        analysis = analyze_code_structure(student_code)

        if "Python Syntax Error" not in analysis["feedback"] and session.attempt_count > 0 and detect_language(student_code) == "python":
            session.reset_attempts()

        # the socratic system prompt
        system_instruction = f"""
        You are a Socratic Teaching Assistant.
        GOAL: Guide the student towards the solution. Give them HINTS.
        RULES: Do NOT write the code for them. NO CODE BLOCKS. NO DIRECT FIXES.

        CURRENT CONTEXT:
        - Student Code: {student_code}
        - Analysis Report: {analysis['feedback']}

        INSTRUCTIONS:
        - You are strictly FORBIDDEN from writing code solutions.
        - If the Analysis Report reports an error OR if you see an obvious syntax error in C++/Java/etc., point it out. Do NOT fix it.
        - If the user asks for code, refuse politely. Give a conceptual hint instead. Do not say "Change X to Y". Ask "What should X be?"
        - Keep answers short (max 3 sentences).
        - Be friendly but firm.
        """

    # we build the message chain for Llama-3 by combining the system instruction + recent chat history
    messages = [{"role": "system", "content": system_instruction}]
    messages.extend(session.stm_window)
    messages.append({"role": "user", "content": user_query})

    # generate response using apply_chat_template from tokenizer
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = text_generator(prompt, max_new_tokens=256, do_sample=True)

    response = outputs[0]["generated_text"][len(prompt):].strip()

    # if the bot slipped up and gave code, we intercept it here
    if not is_theory and has_code_block(response):
        # we feed the "bad" response back to the model and demand a fix
        correction_messages = messages.copy()
        correction_messages.append({"role": "assistant", "content": response})
        correction_messages.append({
            "role": "user",
            "content": "STOP! You violated the rule: NO CODE BLOCKS. Rewrite your previous response immediately. Provide a conceptual hint ONLY. Do not write code."
        })

        # re-generate
        new_prompt = tokenizer.apply_chat_template(correction_messages, tokenize=False, add_generation_prompt=True)
        new_outputs = text_generator(new_prompt, max_new_tokens=256, do_sample=True)
        new_response = new_outputs[0]["generated_text"][len(new_prompt):].strip()

        # fallback if it fails twice
        if has_code_block(new_response):
            return "I'm having trouble giving a hint without showing the answer! Let's just focus on the line number mentioned in the error."

        return new_response

    return response

Initialize session

In [18]:
session = TutorSession()

## Method to start the interactive chat "IDE"

In [ ]:
header = widgets.HTML("<h2>🎓 AI Teaching Assistant 🎓</h2>")

#file upload
upload_btn = widgets.FileUpload(accept='.pdf,.txt', multiple=False, description="Upload Curriculum")
upload_out = widgets.Output()
def on_upload(change):
    for n, f in upload_btn.value.items():
        with open(n, "wb") as w: w.write(f['content'])
        with upload_out: rag_engine.ingest_file(n)
upload_btn.observe(on_upload, names='value')

# editor and chat
code_editor = widgets.Textarea(placeholder='Code goes here...', layout=widgets.Layout(width='98%', height='350px'))
chat_display = widgets.Output(layout={'border': '1px solid #ccc', 'height': '470px', 'overflow_y': 'scroll', 'padding': '10px'})
user_input = widgets.Text(placeholder='Type here...', layout=widgets.Layout(width='60%'))

# difficulty controls
difficulty_dropdown = widgets.Dropdown(
    options=['Any', 'introductory', 'interview', 'competition'],
    value='Any',
    description='Difficulty:',
    layout=widgets.Layout(width='45%')
)

practice_btn = widgets.Button(description="Start Practice Problem 🎲", button_style='primary', icon='code', layout=widgets.Layout(width='50%'))

send_btn = widgets.Button(description="Send", button_style='success', icon='paper-plane')
new_chat_btn = widgets.Button(description="Reset", button_style='warning', icon='refresh')

def log_to_chat(role, text, color="#f0f0f0", align="left"):
    html = f"<div style='text-align:{align}; margin:10px;'><b>{role}:</b><br><span style='background:{color}; padding:8px; border-radius:10px; display:inline-block; white-space:pre-wrap;'>{text}</span></div>"
    with chat_display: display(HTML(html))

def on_send(b):
    msg, code = user_input.value, code_editor.value
    if not msg: return
    log_to_chat("🧑‍🎓 Student", msg, "#e3f2fd", "right")
    user_input.value = ""

    try:
        resp = generate_tutor_response(code, msg, session)
        session.add_turn("user", msg)
        session.add_turn("assistant", resp)
        log_to_chat("🤖 Teaching Assistant", resp)
    except Exception as e:
        log_to_chat("❌ System", str(e), "#ffcdd2")

def on_new_chat(b):
    session.start_new_chat()
    chat_display.clear_output()

def on_practice(b):
    level = difficulty_dropdown.value
    practice_btn.description = f"Searching ({level})..."
    practice_btn.disabled = True

    try:
        prob = exercise_db.get_random_problem(difficulty_level=level)

        if prob:
            session.set_active_problem(prob['question'], prob['solution'])
            chat_display.clear_output()
            code_editor.value = prob['starter_code']

            # Format Difficulty Label
            diff_color = {"introductory": "green", "interview": "orange", "competition": "red", "Any": "blue"}.get(prob['difficulty'], "black")

            intro = f"NEW CHALLENGE <span style='color:{diff_color}'>[{prob['difficulty'].upper()}]</span>\n\n{prob['question']}\n\nI've pasted the starter code."
            log_to_chat("🏆 Practice Mode", intro, "#fff9c4")
        else:
            log_to_chat("⚠️ System", f"Could not find a {level} problem in reasonable time. Try again.", "#fff")

    except Exception as e:
        log_to_chat("❌ Error", f"Failed to load problem: {e}")

    practice_btn.description = "Start Practice Problem 🎲"
    practice_btn.disabled = False

send_btn.on_click(on_send)
new_chat_btn.on_click(on_new_chat)
practice_btn.on_click(on_practice)

controls = widgets.HBox([user_input, send_btn, new_chat_btn], layout=widgets.Layout(width='100%', justify_content='space-between'))
practice_controls = widgets.HBox([difficulty_dropdown, practice_btn], layout=widgets.Layout(width='100%', justify_content='space-between'))

left = widgets.VBox([
    widgets.Label("📚 Curriculum"), upload_btn, upload_out,
    widgets.HTML("<hr>"),
    widgets.Label("🎯 Practice Mode"), practice_controls,
    widgets.HTML("<hr>"),
    widgets.Label("💻 Editor"), code_editor
], layout=widgets.Layout(width='50%', padding='0 10px 0 0'))

right = widgets.VBox([widgets.Label("💬 Chat"), chat_display, controls], layout=widgets.Layout(width='50%'))
display(widgets.VBox([header, widgets.HBox([left, right])]))

Bad code examples to test on

In [20]:
easy_bad_code = "def hello_world()\n    print('Hi')"

harder_bad_code = """
def sum_list(numbers):
    total = 0
    for i in range(len(numbers)):
        # I don't know what to put here to add the numbers
        pass
    return total
"""